# Day 4: Generators and Iterators


## Regular Function vs Generator (Practical Comparison)

This example demonstrates the difference between:
- A regular function that computes all values at once
- A generator that produces values lazily (on demand)


In [1]:
def regular_range(n):
    result = []
    for i in range(n):
        result.append(i)
    return result

print("Calling regular function...")
nums = regular_range(5)
print("Returned value: ",nums)

Calling regular function...
Returned value:  [0, 1, 2, 3, 4]


**What happens here**
- The entire list is created immediately
- All values exist in memory at once

### Generator Function (Lazy Evaluation)

This generator produces values only when requested.


In [2]:
def generator_range(n):
    for i in range(n):
        yield i

print("Calling generator function...")
nums_gen = generator_range(5)
print("Returned object:", nums_gen)

Calling generator function...
Returned object: <generator object generator_range at 0x10a49e260>


**Observation**

- The function does not execute immediately
- A generator object is returned

### Consuming Values from a Generator


In [3]:
print(next(nums_gen))
print(next(nums_gen))
print(next(nums_gen))


0
1
2


### Using a Generator in a Loop


In [4]:
for num in generator_range(5):
    print(num)

0
1
2
3
4


**What happens**
- Values are produced one by one
- Generator is exhausted after the loop

### Key Takeaway

- Regular functions return all results at once
- Generators return values one at a time
- Generators are more memory-efficient
- Generators execute only when needed


## Creating Generators with `yield`

This notebook demonstrates:
- How a generator produces values using `yield`
- How execution pauses and resumes
- How generator state is preserved


In [11]:
def countdown(n):
    """Generator that counts down from n"""
    while n>0:
        print(f"In Generator yield {n}")
        yield n
        n -= 1
        print(f"In generator decrement n and now n = {n}")

gen = countdown(5)


**Calling using loop**

In [13]:
for num in countdown(5):
    print(num)

In Generator yield 5
5
In generator decrement n and now n = 4
In Generator yield 4
4
In generator decrement n and now n = 3
In Generator yield 3
3
In generator decrement n and now n = 2
In Generator yield 2
2
In generator decrement n and now n = 1
In Generator yield 1
1
In generator decrement n and now n = 0


**Observation**
- Values are produced one at a time
- The function does not run fully at once

**Calling using next**

In [12]:
print(next(gen))

In Generator yield 5
5


>> The decrement after yield is not called immediately. It will be called next time - see below

In [14]:
print(next(gen))

In generator decrement n and now n = 4
In Generator yield 4
4


### How `yield` Pauses and Resumes Execution


In [16]:
def simple_generator():
    print("First yield")
    yield 1
    print("Second yield")
    yield 2
    print("Third yield")
    yield 3

gen = simple_generator()

**First `next()` Call**

In [18]:
next(gen)

First yield


1

**What happens**

-   Function starts execution
    
-   Stops at first `yield`
    
-   Returns `1`

**Second `next()` Call**

In [19]:
print(next(gen))

Second yield
2


**What happens**

-   Execution resumes after the first `yield`
    
-   Stops at the second `yield`
    
-   Returns `2`

**Third `next()` Call**

In [20]:
print(next(gen))

Third yield
3


**What happens**

-   Execution resumes again
    
-   Stops at the third `yield`
    
-   Returns `3`

## Generator Expressions

This section demonstrates:
- The difference between list comprehensions and generator expressions
- Memory-efficient iteration
- Lazy evaluation in practice


### List Comprehension (Eager Evaluation)

In [21]:
squares_list = [x**2 for x in range(20)]

print("Type:", type(squares_list))
print("Values:", squares_list)

Type: <class 'list'>
Values: [0, 1, 4, 9, 16, 25, 36, 49, 64, 81, 100, 121, 144, 169, 196, 225, 256, 289, 324, 361]


**Observation**

-   All values are created immediately
    
-   Stored entirely in memory
    

### Generator Expression (Lazy Evaluation)


In [22]:
squares_gen = (x**2 for x in range(20))

print("Type:", type(squares_gen))
print("Generator object:", squares_gen)

Type: <class 'generator'>
Generator object: <generator object <genexpr> at 0x1102452f0>


**Observation**

-   No values computed yet
    
-   Only a generator object is created
    

### Consuming Values from a Generator Expression



**Iteration**

In [24]:
for square in squares_gen:
    print(square)

0
1
4
9
16
25
36
49
64
81
100
121
144
169
196
225
256
289
324
361


**Observation**

-   Values are produced one at a time
    
-   Generator is exhausted after iteration
    

### Early Termination (Memory Efficiency)



In [25]:
squares_gen = (x**2 for x in range(1000000))

for square in squares_gen:
    print(square)
    if square > 100:
        break

0
1
4
9
16
25
36
49
64
81
100
121


**What this shows**

-   Values stop generating as soon as the condition is met
-   Remaining values are never computed

### Key Takeaways

- Generator expressions use parentheses `()`
- They evaluate lazily
- Memory usage stays low
- Ideal for large datasets and pipelines



## Practical Examples of Generators

This notebook demonstrates real-world use cases of generators:
- Reading large files efficiently
- Working with infinite sequences
- Building data-processing pipelines


### 1. Reading Large Files (Line by Line)

In [29]:
def read_large_file(filepath):
    """Memory-efficient file reading"""
    with open(filepath) as f:
        for line in f:
            yield line.strip()

> Note: For demonstration, we will create a small file instead of a huge one.



#### Create a Sample File

In [27]:
with open("file.txt", "w") as f:
    f.write("line 1\n")
    f.write("line 2\n")
    f.write("line 3\n")

#### Process File Using Generator

In [30]:
for line in read_large_file("file.txt"):
    print(line)

line 1
line 2
line 3


**Observation**

-   Lines are read one at a time
    
-   The entire file is never loaded into memory
    


### 2. Infinite Sequences Using Generators

Generators can represent sequences that do not have a natural end.
Values are produced only when requested.



In [31]:
def fibonacci():
    """Infinite Fibonacci sequence"""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a+b

#### Consume Finite Values from Infinite Generator

In [32]:
fib = fibonacci()

for _ in range(10):
    print(next(fib))

0
1
1
2
3
5
8
13
21
34


**Observation**

-   The generator does not stop by itself
    
-   The consumer controls how many values are generated
   

### 3. Generator Pipeline Processing

Generators can be chained together to process data step by step,
without creating intermediate lists.



In [33]:
def read_data(filename):
    """Read lines from a file"""
    with open(filename) as f:
        for line in f:
            yield line.strip()

def filter_comments(lines):
    """Remove comment lines"""
    for line in lines:
        if not line.startswith("#"):
            yield line

def parse_numbers(lines):
    """Convert lines to integers"""
    for line in lines:
        try:
            yield int(line)
        except ValueError:
            pass


#### Create Sample Data File

In [34]:
with open("data.txt", "w") as f:
    f.write("# comment\n")
    f.write("10\n")
    f.write("20\n")
    f.write("invalid\n")
    f.write("30\n")

#### Chain Generators

In [35]:
numbers = parse_numbers(filter_comments(read_data("data.txt")))
total = sum(numbers)

print("Total: ", total)


Total:  60


**Observation**

-   Data flows through multiple generators
    
-   No intermediate data structures are created
    
-   Processing is memory-efficient
    


### Key Takeaways

- Generators process data lazily
- Ideal for large files and streams
- Infinite sequences are safe when controlled
- Generator pipelines are clean and efficient

## The Iterator Protocol

This section demonstrates:
- How the iterator protocol works
- How to create a custom iterator using `__iter__` and `__next__`
- How iterators behave in a `for` loop



### Custom Iterator Implementation

In [36]:
class CountDown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self
    
    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        self.current -= 1
        return self.current + 1

### Using the Custom Iterator


In [37]:
for num in CountDown(5):
    print(num)

5
4
3
2
1



**Observation**

-   The `for` loop calls `__iter__()` once
    
-   `__next__()` is called repeatedly
    
-   Iteration stops automatically when `StopIteration` is raised
    

## Generator Delegation with `yield from`

This section shows how `yield from` delegates iteration to another generator
and simplifies generator composition.



### Individual Generators


In [38]:
def generator1():
    yield 1
    yield 2

def generator2():
    yield 3
    yield 4

### Combining Generator Using `yield from`

In [39]:
def combined():
    yield from generator1()
    yield from generator2()

### Consuming the Combined Generator


In [41]:
list(combined())

[1, 2, 3, 4]

### Key Takeaways

- Iterators implement `__iter__()` and `__next__()`
- Generators automatically follow the iterator protocol
- `yield from` simplifies generator composition
- Cleaner and more readable than manual loops


## Generator Methods: `send()` and `close()`

This section demonstrates:
- How to send values into a generator using `send()`
- How to stop a generator using `close()`
- How generator execution behaves step by step



###  Generator Using `send()`

In [42]:
def echo_generator():
    while True:
        value = yield
        print(f"Received: {value}")

### Creating and Priming the Generator

In [43]:
gen = echo_generator()

# Prime the generator so it reaches the first `yield`
next(gen)

**Observation**

-   The generator is now paused at `yield`
    
-   Ready to receive values using `send()`
    


### Sending Values into the Generator



In [44]:
gen.send("Hello")
gen.send("World")

Received: Hello
Received: World


**Observation**

-   Each `send()` resumes execution
    
-   The sent value becomes the result of the `yield` expression
    

## Stopping a Generator with `close()`



### Infinite Counter Generator

In [45]:
def counter():
    n = 0
    while True:
        yield n
        n += 1

### Consuming and Closing the Generator



In [46]:
gen = counter()

print(next(gen))
print(next(gen))

# Stog the generator explicitly
gen.close()

0
1


### Generator After `close()`


In [47]:
# This will raise StopIteration
next(gen)

StopIteration: 

### Key Takeaways

- `send()` allows data to be passed into a generator
- Generators must be primed before calling `send()`
- `close()` terminates a generator immediately
- Once closed, a generator cannot be resumed



## Performance Benefits of Generators

This section compares the memory usage of:
- A list (eager evaluation)
- A generator (lazy evaluation)

The goal is to understand why generators are more memory-efficient.


### List Memory Usage

In [48]:
import sys

numbers_list = [x for x in range(1000000)]
print("List type:", type(numbers_list))
print("List size:", sys.getsizeof(numbers_list), "bytes")

List type: <class 'list'>
List size: 8448728 bytes


**Observation**

-   The list stores all values in memory
    
-   Memory usage increases with dataset size
    


### Generator Memory Usage

In [49]:
numbers_gen = (x for x in range(1000000))
print("Generator type:", type(numbers_gen))
print("Generator size:", sys.getsizeof(numbers_gen), "bytes")

Generator type: <class 'generator'>
Generator size: 200 bytes


**Observation**

-   Only the generator object exists in memory
    
-   Values are generated on demand
    


### Stop Early with Generator


In [50]:
numbers_gen = (x for x in range(1000000))

for num in numbers_gen:
    if num > 10:
        break

**What this shows**

-   Generator stops producing values early
    
-   Remaining values are never computed or stored
    

### Key Takeaways

- Lists allocate memory for all elements upfront
- Generators use constant memory
- Generators support early termination efficiently
- Ideal for large or streaming data

